---

# Tugas Besar 2 IF3170 Inteligensi Artifisial

*This notebook is the assignment template. Each group works on its own copy, and cells or sections may be added as needed.*

---

## Group 28 

- Varistha Devi (1524135)
- Mirza Tsabita Wafa'ana (NIM)
- Eliana Natalie Widjojo (13524116)
- A. Rinofaros (13524138)
- Safira Berlianti (13524128)

---

## Table of Contents

1. [**Introduction**](#1)
2. [**Initialization**](#2)
3. [**Data Understanding (Optional)**](#3)
4. [**Training and Validation Split**](#4)
5. [**Data Preparation**](#5)
6. [**Preparation Pipeline**](#6)
7. [**Modeling and Validation**](#7)
8. [**Comparison with scikit-learn**](#8)
9. [**Saving and Loading Models**](#9)
10. [**Kaggle Submission**](#10)
11. [**Error Analysis**](#11)


---

# Introduction <a name="1"></a>

---

This section frames the problem in the words of the group.

## Overview

*To be written by the group.*

## Aim

*To be written by the group.*

## Metric

*To be written by the group.*

## Dataset

*To be written by the group.*

## Approach

*To be written by the group.*

---

# Initialization <a name="2"></a>

---

This section prepares the environment, the settings, and the data files.

## Environment Setup

The following cell lists the base libraries. The install line is commented by default.

In [1]:
# %pip install -q numpy pandas

## Import Libraries

Every import of the notebook is kept in this cell.

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)


from ..src.knn.knn import KNN
from ..src.svm.svm import SVM

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Settings

Paths and shared constants are kept in this class.

In [ ]:
class Settings:
    SEED = 42
    DATA_DIR = Path("data")
    OUTPUT_DIR = Path("output")
    TARGET = "is_canceled"
    VALIDATION_SIZE = 0.20
    OUTLIER_IQR_FACTOR = 1.5
    APPLY_PCA = False
    PCA_VARIANCE = 0.95
    APPLY_RANDOM_OVERSAMPLING = False

CFG = Settings()
CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Seed Everything

A fixed seed keeps the results reproducible.

In [ ]:
def seed_everything(seed: int):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(CFG.SEED)

## Load Dataset

The competition files are read here. The `id` column stays as text because some values resemble numbers.

In [ ]:
train_path = CFG.DATA_DIR / "train.csv"
test_path = CFG.DATA_DIR / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(
        f"File {train_path} tidak ditemukan. Letakkan train.csv di folder data/."
    )
if not test_path.exists():
    raise FileNotFoundError(
        f"File {test_path} tidak ditemukan. Letakkan test.csv di folder data/."
    )

train_df = pd.read_csv(train_path, dtype={"id": str})
test_df = pd.read_csv(test_path, dtype={"id": str})

print(f"Train: {train_df.shape}")
print(f"Test : {test_df.shape}")
display(train_df.head())

---

# Data Understanding (Optional) <a name="3"></a>

---

Data understanding builds the knowledge that later preparation decisions rest on.

## Describe Data

This part covers the size, the attribute types, and the meaning of each column.

In [ ]:
print("Ukuran dataset:", train_df.shape)
print("\nTipe data setiap atribut:")
display(train_df.dtypes.rename("dtype").to_frame())

print("\nInformasi dataset:")
train_df.info()

print("\nStatistik deskriptif:")
display(train_df.describe(include="all").T)

## Explore Data

This part covers distributions and relationships between attributes.

In [ ]:
# Distribusi target
if CFG.TARGET in train_df.columns:
    display(train_df[CFG.TARGET].value_counts(dropna=False).to_frame("jumlah"))
    display(
        train_df[CFG.TARGET]
        .value_counts(normalize=True, dropna=False)
        .mul(100)
        .round(2)
        .to_frame("persentase (%)")
    )

# Distribusi beberapa atribut numerik
numeric_preview = train_df.select_dtypes(include=np.number).columns[:6]
if len(numeric_preview) > 0:
    train_df[numeric_preview].hist(figsize=(14, 8), bins=30)
    plt.tight_layout()
    plt.show()

# Korelasi atribut numerik (termasuk target angka)
numeric_df = train_df.select_dtypes(include=np.number)
if numeric_df.shape[1] > 1:
    plt.figure(figsize=(12, 9))
    sns.heatmap(numeric_df.corr(), cmap="coolwarm", center=0)
    plt.title("Korelasi Atribut Numerik")
    plt.tight_layout()
    plt.show()

## Verify Data Quality

This part covers completeness, consistency, and the validity of values.

In [ ]:
print("Jumlah missing values per kolom:")
missing_report = train_df.isna().sum()
display(
    missing_report[missing_report > 0]
    .sort_values(ascending=False)
    .to_frame("missing_values")
)

print("Jumlah baris duplikat:", train_df.duplicated().sum())

if CFG.TARGET in train_df.columns:
    print("\nDistribusi target:")
    print(train_df[CFG.TARGET].value_counts(dropna=False))

---

# Training and Validation Split <a name="4"></a>

---

The split comes before data preparation so that validation data stays unseen. Only `train.csv` is split, and `test.csv` serves the Kaggle submission alone.

In [ ]:
if CFG.TARGET not in train_df.columns:
    raise KeyError(f"Kolom target '{CFG.TARGET}' tidak ditemukan di train.csv.")

# Split dilakukan sebelum fitting preprocessing agar validation tetap unseen
train_set, val_set = train_test_split(
    train_df,
    test_size=CFG.VALIDATION_SIZE,
    random_state=CFG.SEED,
    stratify=train_df[CFG.TARGET]
)

train_set = train_set.copy().reset_index(drop=True)
val_set = val_set.copy().reset_index(drop=True)

print(f"Training set   : {train_set.shape}")
print(f"Validation set : {val_set.shape}")
print("\nDistribusi target training:")
print(train_set[CFG.TARGET].value_counts(normalize=True).sort_index())
print("\nDistribusi target validation:")
print(val_set[CFG.TARGET].value_counts(normalize=True).sort_index())

---

# Data Preparation <a name="5"></a>

---

Each process below applies where the data calls for it, and the Insights block under it records the reason for the decision.

Preparation steps may use `sklearn`. Every statistic comes from the training split only, then transforms the other sets.

## Select Data

Selection decides which records and attributes stay in the analysis.

In [ ]:
target_col = CFG.TARGET

#  identifier disimpan jika ada dan tidak digunakan sebagai fitur
id_col = "id" if "id" in train_set.columns else None
train_ids = train_set[id_col].copy() if id_col else None
val_ids = val_set[id_col].copy() if id_col else None
test_ids = test_df[id_col].copy() if id_col and id_col in test_df.columns else None

# Kolom yang menunjukan hasil akhir reservasi mungkin buat data leakage
leakage_candidates = [
    target_col,
    "reservation_status",
    "reservation_status_date",
]
drop_cols = [c for c in leakage_candidates + ([id_col] if id_col else []) if c and c in train_set.columns]

y_train = train_set[target_col].copy()
y_val = val_set[target_col].copy()
X_train = train_set.drop(columns=drop_cols, errors="ignore").copy()
X_val = val_set.drop(columns=drop_cols, errors="ignore").copy()

if not test_df.empty:
    X_test = test_df.drop(
        columns=[c for c in leakage_candidates + ([id_col] if id_col else []) if c in test_df.columns],
        errors="ignore"
    ).copy()
else:
    X_test = pd.DataFrame(columns=X_train.columns)

#  fitur test disamakan dengan training, nanti imputasi menangani kolom yg tidak tersedia
X_test = X_test.reindex(columns=X_train.columns)

print("Kolom yang dihapus:", drop_cols)
print("Fitur training:", X_train.shape)
print("Fitur validation:", X_val.shape)
print("Fitur test:", X_test.shape)

## Clean Data

Cleaning raises data quality to the level the chosen models need.

### Missing Values

This part covers missing values, if any.

In [ ]:
# Kolom yang seluruh nilainya kosong dan tidak memberikan informasi
all_missing_cols = X_train.columns[X_train.isna().all()].tolist()
print("Kolom seluruhnya missing:", all_missing_cols)

X_train.drop(columns=all_missing_cols, inplace=True)
X_val.drop(columns=all_missing_cols, inplace=True)
X_test.drop(columns=all_missing_cols, inplace=True)

# Nilai kosong pada kolom lain tidak diisi secara manual di sini.
# Imputasi median/modus akan di-fit hanya pada training set di preparation pipeline.
missing_train = X_train.isna().sum()
display(missing_train[missing_train > 0].sort_values(ascending=False).to_frame("missing_train"))

### Duplicates

This part covers repeated records.

In [ ]:
# Duplikasi hanya dihapus dari training set. Validation/test tidak dipakai untuk menentukan baris training yang dipertahankan.
duplicate_count = X_train.duplicated().sum()
print("Duplikasi pada training sebelum penghapusan:", duplicate_count)

if duplicate_count:
    keep_mask = ~X_train.duplicated(keep="first")
    X_train = X_train.loc[keep_mask].reset_index(drop=True)
    y_train = y_train.loc[keep_mask].reset_index(drop=True)

print("Ukuran training setelah penghapusan duplikasi:", X_train.shape)

### Outliers

This part covers values far from the rest of the data.

In [ ]:

# Deteksi batas outlier dengan metode IQR pada training set.
# Batas diterapkan ke validation dan test agar tidak terjadi data leakage.
numeric_for_outlier = X_train.select_dtypes(include=np.number).columns.tolist()
outlier_bounds = {}

for col in numeric_for_outlier:
    q1 = X_train[col].quantile(0.25)
    q3 = X_train[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - CFG.OUTLIER_IQR_FACTOR * iqr
    upper = q3 + CFG.OUTLIER_IQR_FACTOR * iqr
    outlier_bounds[col] = (lower, upper)

    n_outliers = ((X_train[col] < lower) | (X_train[col] > upper)).sum()
    if n_outliers:
        print(f"{col}: {n_outliers} outlier")

def clip_outliers(frame, bounds):
    frame = frame.copy()
    for col, (lower, upper) in bounds.items():
        if col in frame.columns:
            frame[col] = frame[col].clip(lower=lower, upper=upper)
    return frame

X_train = clip_outliers(X_train, outlier_bounds)
X_val = clip_outliers(X_val, outlier_bounds)
X_test = clip_outliers(X_test, outlier_bounds)here

### Data Errors

This part covers wrong types, inconsistent labels, and impossible values.

In [ ]:
def correct_data_errors(frame):
    frame = frame.copy()

    # Rapikan whitespace pada atribut kategorikal.
    categorical = frame.select_dtypes(include=["object", "category"]).columns
    for col in categorical:
        frame[col] = frame[col].map(
            lambda value: value.strip() if isinstance(value, str) else value
        )

    non_negative_candidates = [
        "adults", "children", "babies",
        "stays_in_weekend_nights", "stays_in_week_nights",
        "previous_cancellations", "previous_bookings_not_canceled",
        "booking_changes", "days_in_waiting_list",
        "required_car_parking_spaces", "total_of_special_requests",
        "adr",
    ]
    for col in non_negative_candidates:
        if col in frame.columns:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
            frame.loc[frame[col] < 0, col] = np.nan

    return frame

X_train = correct_data_errors(X_train)
X_val = correct_data_errors(X_val)
X_test = correct_data_errors(X_test)

print("Pembersihan tipe dan nilai tidak valid selesai.")

## Construct Data

Construction derives attributes and transforms values for modeling.

### Attribute Construction

<!-- rubric v3: derived columns, rate_per_guest_night -->

This part covers new attributes derived from existing ones.

In [ ]:
def construct_attributes(frame):
    frame = frame.copy()

    guest_cols = [c for c in ["adults", "children", "babies"] if c in frame.columns]
    night_cols = [c for c in ["stays_in_weekend_nights", "stays_in_week_nights"] if c in frame.columns]

    if guest_cols:
        frame["total_guests"] = frame[guest_cols].fillna(0).sum(axis=1)

    if night_cols:
        frame["total_nights"] = frame[night_cols].fillna(0).sum(axis=1)

    if "total_guests" in frame.columns and "total_nights" in frame.columns:
        denominator = frame["total_guests"] * frame["total_nights"]
        if "adr" in frame.columns:
            frame["rate_per_guest_night"] = (
                frame["adr"] / denominator.replace(0, np.nan)
            )

    return frame

X_train = construct_attributes(X_train)
X_val = construct_attributes(X_val)
X_test = construct_attributes(X_test)

print("Fitur setelah konstruksi:", X_train.shape[1])
display(X_train.head())

### Feature Scaling

This part covers normalization to the range 0 to 1 and standardization.

In [ ]:
# Identifikasi fitur numerik dan pipeline scaling.
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

print(f"Jumlah fitur numerik: {len(numerical_cols)}")
print("Contoh fitur numerik:", numerical_cols[:10])

### Encoding

This part covers the numeric representation of categorical attributes.

In [ ]:
# One-hot encoding
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

try:
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # Kompatibilitas dengan scikit-learn versi lama
    one_hot = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", one_hot),
])

print(f"Jumlah fitur kategorikal: {len(categorical_cols)}")
print("Contoh fitur kategorikal:", categorical_cols[:10])

### Dimensionality Reduction

This part covers fewer dimensions that keep the meaning of the data.

In [ ]:
# Secara default dinonaktifkan agar interpretabilitas fitur hasil encoding tetap terjaga dan semua model menerima fitur yg sama
if CFG.APPLY_PCA:
    dimensionality_reducer = PCA(
        n_components=CFG.PCA_VARIANCE,
        random_state=CFG.SEED
    )
else:
    dimensionality_reducer = None

print("PCA aktif:", dimensionality_reducer is not None)

## Class Balance

This part covers the proportion of each class in the training data.

In [ ]:
print("Distribusi kelas pada training:")
class_counts = y_train.value_counts().sort_index()
class_ratio = y_train.value_counts(normalize=True).sort_index()
display(pd.DataFrame({
    "jumlah": class_counts,
    "proporsi": class_ratio
}))

print("Distribusi kelas pada validation (tidak diubah):")
display(y_val.value_counts(normalize=True).sort_index().to_frame("proporsi"))

## Format Data

Formatting produces the structure the models read, such as a numeric matrix.

In [ ]:
# target dipastikan memiliki nilai dan fitur train/validation/test dengan kolom sama.
if y_train.isna().any() or y_val.isna().any():
    raise ValueError("Target mengandung missing values; periksa dataset sebelum modeling.")

# Urutan fitur disamakan sebelum transformasi.
X_val = X_val.reindex(columns=X_train.columns)
X_test = X_test.reindex(columns=X_train.columns)

print("Target training:", y_train.shape)
print("Target validation:", y_val.shape)
print("Jumlah fitur sebelum encoding:", X_train.shape[1])

---

# Preparation Pipeline <a name="6"></a>

---

<!-- rubric v3: fitted steps, fitted_prep_steps -->

The pipeline combines the preparation steps so they run in one consistent order.

In [ ]:
# imputasi, scaling, dan encoding digabungkan dalam pipeline pre-processing.
transformers = []
if numerical_cols:
    transformers.append(("num", numeric_transformer, numerical_cols))
if categorical_cols:
    transformers.append(("cat", categorical_transformer, categorical_cols))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)

# Fit hanya pada training; validation dan test hanya ditransformasi.
X_train_array = preprocessor.fit_transform(X_train)
X_val_array = preprocessor.transform(X_val)
X_test_array = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_array, columns=feature_names, index=X_train.index)
X_val_processed = pd.DataFrame(X_val_array, columns=feature_names, index=X_val.index)
X_test_processed = pd.DataFrame(X_test_array, columns=feature_names)

# fit pada training saja.
if dimensionality_reducer is not None:
    train_reduced = dimensionality_reducer.fit_transform(X_train_processed)
    val_reduced = dimensionality_reducer.transform(X_val_processed)
    test_reduced = dimensionality_reducer.transform(X_test_processed)

    component_names = [f"PC{i+1}" for i in range(train_reduced.shape[1])]
    X_train_processed = pd.DataFrame(train_reduced, columns=component_names)
    X_val_processed = pd.DataFrame(val_reduced, columns=component_names)
    X_test_processed = pd.DataFrame(test_reduced, columns=component_names)

# Random oversampling opsional dan hanya diterapkan pada training set.
# Validation set dibuat tidak seimbangkan agar evaluasi merepresentasikan distribusi asli.
if CFG.APPLY_RANDOM_OVERSAMPLING:
    counts = y_train.value_counts()
    majority_count = counts.max()
    sampled_indices = []
    rng = np.random.default_rng(CFG.SEED)

    for label, count in counts.items():
        label_indices = np.flatnonzero(y_train.to_numpy() == label)
        sampled_indices.extend(label_indices.tolist())
        if count < majority_count:
            sampled_indices.extend(
                rng.choice(label_indices, size=majority_count-count, replace=True).tolist()
            )

    rng.shuffle(sampled_indices)
    X_train_processed = X_train_processed.iloc[sampled_indices].reset_index(drop=True)
    y_train_processed = y_train.iloc[sampled_indices].reset_index(drop=True)
else:
    y_train_processed = y_train.reset_index(drop=True)

y_val_processed = y_val.reset_index(drop=True)

print("Training processed  :", X_train_processed.shape, y_train_processed.shape)
print("Validation processed:", X_val_processed.shape, y_val_processed.shape)
print("Test processed      :", X_test_processed.shape)

---

# Modeling and Validation <a name="7"></a>

---

The target is `is_canceled`. The models here are implemented from scratch, with math libraries such as NumPy.

## K-Nearest Neighbors

<!-- rubric v3: pairwise distances, val_neighbor_dists -->

This part covers the nearest neighbor classifier and its validation.

In [ ]:
# konversi DataFrame hasil preprocessing menjadi NumPy array.
X_train_knn = X_train_processed.to_numpy(dtype=float)
X_val_knn = X_val_processed.to_numpy(dtype=float)
y_train_knn = y_train_processed.to_numpy()
y_val_knn = y_val_processed.to_numpy()

# default KNN.
KNN_K = 5
KNN_DISTANCE_METRIC = "euclidean"  # euclidean, manhattan, minkowski
KNN_WEIGHTS = "uniform"            # uniform, distance

knn_model = KNN(
    k=KNN_K,
    distance_metric=KNN_DISTANCE_METRIC,
    weights=KNN_WEIGHTS,
)

# Fit hanya pakai data training hasil preprocessing.
knn_model.fit(X_train_knn, y_train_knn)

# Prediksi dan evaluasi pakai validation set.
y_val_pred_knn = knn_model.predict(X_val_knn)

knn_metrics = {
    "Accuracy": accuracy_score(y_val_knn, y_val_pred_knn),
    "Precision (macro)": precision_score(
        y_val_knn, y_val_pred_knn, average="macro", zero_division=0
    ),
    "Recall (macro)": recall_score(
        y_val_knn, y_val_pred_knn, average="macro", zero_division=0
    ),
    "F1-score (macro)": f1_score(
        y_val_knn, y_val_pred_knn, average="macro", zero_division=0
    ),
}

print(f"KNN configuration: k={KNN_K}, metric={KNN_DISTANCE_METRIC}, weights={KNN_WEIGHTS}")
display(pd.Series(knn_metrics, name="Validation Score").to_frame())

print("\\nClassification Report:")
print(classification_report(y_val_knn, y_val_pred_knn, zero_division=0))

print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_val_knn, y_val_pred_knn),
    index=[f"Actual {label}" for label in knn_model.classes_],
    columns=[f"Predicted {label}" for label in knn_model.classes_],
))


## Decision Tree Learning

This part covers one chosen variant, which is ID3, C4.5, or CART, and its validation.

In [ ]:
# Write your code here

## Logistic Regression

This part covers the model with L1, L2, and Elastic Net regularization.

In [ ]:
# Write your code here

## Support Vector Machine

<!-- rubric v3: total slack, slack_total -->

This part covers the linear, polynomial, and Gaussian kernels.

In [ ]:
# Konversi DataFrame hasil preprocessing menjadi NumPy array 
X_train_svm = X_train_processed.to_numpy(dtype=float)
X_val_svm = X_val_processed.to_numpy(dtype=float)
y_train_svm = y_train_processed.to_numpy()
y_val_svm = y_val_processed.to_numpy()

# Eksperimen untuk ketiga kernel: Linear, Polynomial, dan Gaussian
svm_kernels = ["linear", "poly", "rbf"]
svm_models = {}
svm_results = []

for kernel_name in svm_kernels:
    svm_model = SVM(
        kernel=kernel_name,
        C=1.0,
        degree=3,
        gamma="scale",
        coef0=1.0,
        lr=0.01 if kernel_name == "linear" else 0.005,
        epochs=50,
        max_kernel_samples=2000,  # supaya aman untuk RAM 
        random_state=CFG.SEED,
    )

    # Fit hanya menggunakan data training hasil preprocessing
    svm_model.fit(X_train_svm, y_train_svm)
    svm_models[kernel_name] = svm_model

    # Prediksi dan evaluasi menggunakan validation set
    y_val_pred_svm = svm_model.predict(X_val_svm)

    # Variabel total slack sesuai petunjuk rubrik
    slack_total = svm_model.slack_total

    svm_results.append({
        "Kernel": kernel_name,
        "Accuracy": accuracy_score(y_val_svm, y_val_pred_svm),
        "Precision (macro)": precision_score(
            y_val_svm, y_val_pred_svm, average="macro", zero_division=0
        ),
        "Recall (macro)": recall_score(
            y_val_svm, y_val_pred_svm, average="macro", zero_division=0
        ),
        "F1-score (macro)": f1_score(
            y_val_svm, y_val_pred_svm, average="macro", zero_division=0
        ),
        "Total Slack": slack_total,
    })

    print(f"=== SVM ({kernel_name.upper()}) | Total Slack: {slack_total:.2f} ===")
    print(classification_report(y_val_svm, y_val_pred_svm, zero_division=0))

# Summary perbandingan
svm_summary_df = pd.DataFrame(svm_results).set_index("Kernel")
display(svm_summary_df)

# Pilih model SVM terbaik berdasarkan F1-score untuk evaluasi & confusion matrix
best_svm_kernel = svm_summary_df["F1-score (macro)"].idxmax()
best_svm_model = svm_models[best_svm_kernel]
y_val_pred_svm = best_svm_model.predict(X_val_svm)

print(f"\nBest SVM Kernel: {best_svm_kernel.upper()}")
print("Confusion Matrix:")
display(pd.DataFrame(
    confusion_matrix(y_val_svm, y_val_pred_svm),
    index=[f"Actual {label}" for label in best_svm_model.classes_],
    columns=[f"Predicted {label}" for label in best_svm_model.classes_],
))

## Naive Bayes

This part covers the Naive Bayes classifier and its validation.

In [ ]:
# Write your code here

## Model Evaluation

<!-- figure palette: #6d2fa3 -->

This part compares the validation results of all models.

In [ ]:
# Write your code here

## Improvements (Optional)

This part covers hyperparameter tuning, cross-validation, and further visualization.

In [ ]:
# Write your code here

---

# Comparison with scikit-learn <a name="8"></a>

---

This section compares each from-scratch model with its `scikit-learn` counterpart.

## K-Nearest Neighbors

In [ ]:
# Konversi fitur validation dan training sudah dilakukan pada bagian KNN.
# Gunakan konfigurasi yang sama agar perbandingan adil.
sklearn_knn = KNeighborsClassifier(
    n_neighbors=KNN_K,
    metric=KNN_DISTANCE_METRIC,
    weights=KNN_WEIGHTS,
    p=3 if KNN_DISTANCE_METRIC == "minkowski" else 2,
    n_jobs=-1,
)

# Fit hanya menggunakan data training hasil preprocessing.
sklearn_knn.fit(X_train_knn, y_train_knn)

# Prediksi dan evaluasi pada validation set.
sklearn_knn_preds = sklearn_knn.predict(X_val_knn)

sklearn_knn_results = [{
    "Model": "KNN",
    "Scratch F1 (macro)": knn_metrics["F1-score (macro)"],
    "Sklearn F1 (macro)": f1_score(
        y_val_knn, sklearn_knn_preds, average="macro", zero_division=0
    ),
    "Scratch Accuracy": knn_metrics["Accuracy"],
    "Sklearn Accuracy": accuracy_score(y_val_knn, sklearn_knn_preds),
}]

# Summary perbandingan, mengikuti format tabel perbandingan SVM.
display(pd.DataFrame(sklearn_knn_results).set_index("Model"))

# Laporan klasifikasi dan confusion matrix dari implementasi scikit-learn.
print("Classification Report (scikit-learn KNN):")
print(classification_report(y_val_knn, sklearn_knn_preds, zero_division=0))

print("Confusion Matrix (scikit-learn KNN):")
display(pd.DataFrame(
    confusion_matrix(y_val_knn, sklearn_knn_preds),
    index=[f"Actual {label}" for label in sklearn_knn.classes_],
    columns=[f"Predicted {label}" for label in sklearn_knn.classes_],
))


## Support Vector Machine

In [ ]:
sklearn_svm_results = []

# Compare ketiga kernel
rng = np.random.default_rng(CFG.SEED)
if len(X_train_svm) > 2000:
    sub_idx = rng.choice(len(X_train_svm), size=2000, replace=False)
    X_tr_sk, y_tr_sk = X_train_svm[sub_idx], y_train_svm[sub_idx]
else:
    X_tr_sk, y_tr_sk = X_train_svm, y_train_svm

for kernel_name in ["linear", "poly", "rbf"]:
    if kernel_name == "linear":
        sk_svm = LinearSVC(C=1.0, random_state=CFG.SEED, max_iter=2000)
        sk_svm.fit(X_train_svm, y_train_svm)
    else:
        sk_svm = SVC(kernel=kernel_name, C=1.0, degree=3, gamma="scale", random_state=CFG.SEED)
        sk_svm.fit(X_tr_sk, y_tr_sk)

    sk_preds = sk_svm.predict(X_val_svm)
    sklearn_svm_results.append({
        "Kernel": kernel_name,
        "Scratch F1 (macro)": svm_summary_df.loc[kernel_name, "F1-score (macro)"],
        "Sklearn F1 (macro)": f1_score(y_val_svm, sk_preds, average="macro", zero_division=0),
        "Scratch Accuracy": svm_summary_df.loc[kernel_name, "Accuracy"],
        "Sklearn Accuracy": accuracy_score(y_val_svm, sk_preds),
    })

display(pd.DataFrame(sklearn_svm_results).set_index("Kernel"))

---

# Saving and Loading Models <a name="9"></a>

---

<!-- artifacts: <model>_v3.pkl -->

This section stores the trained models and restores them for prediction.

In [ ]:
# ===== Saving and Loading Models =====
# Semua model hasil training disimpan ke output/models/ memakai pickle
# (KNN.save_model / SVM.save_model), lalu dimuat kembali dan diverifikasi.
MODEL_DIR = CFG.OUTPUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Daftar model yang sudah dilatih di bagian Modeling.
# Tambahkan model lain (DTL, Logistic Regression, Naive Bayes) setelah
# class-nya punya save_model/load_model yang sama.
models_to_save = {
    "knn": (knn_model, KNN, X_val_knn),
    **{f"svm_{k}": (m, SVM, X_val_svm) for k, m in svm_models.items()},
}

save_report = []
for name, (model, model_cls, X_check) in models_to_save.items():
    path = MODEL_DIR / f"{name}_model.pkl"

    # Save
    model.save_model(str(path))

    # Load
    loaded = model_cls.load_model(str(path))

    # Verifikasi: prediksi model hasil load harus identik dengan model asli
    same = bool(np.array_equal(model.predict(X_check), loaded.predict(X_check)))
    save_report.append({
        "Model": name,
        "File": str(path),
        "Ukuran (KB)": round(path.stat().st_size / 1024, 1),
        "Prediksi identik": same,
    })

display(pd.DataFrame(save_report).set_index("Model"))
assert all(r["Prediksi identik"] for r in save_report), "Ada model yang hasil load-nya berbeda!"


---

# Kaggle Submission <a name="10"></a>

---

The final pipeline and model are fitted on the original training set, and the test set is only transformed. The submission file contains the `id` and `is_canceled` columns.

In [ ]:
# ===== Kaggle Submission =====
# Pipeline preprocessing dan model final di-fit ulang pada SELURUH train.csv,
# sedangkan test.csv hanya ditransformasi (tidak digabung dengan train).
FINAL_MODEL = "knn"   # "knn" atau "svm_linear" / "svm_poly" / "svm_rbf"
SUBMISSION_PATH = CFG.OUTPUT_DIR / "real_submission.csv"

seed_everything(CFG.SEED)

# 1) Select data
y_full = train_df[CFG.TARGET].copy()
X_full = train_df.drop(columns=drop_cols, errors="ignore").copy()
X_sub = test_df.drop(columns=drop_cols, errors="ignore").copy()
X_sub = X_sub.reindex(columns=X_full.columns)
sub_ids = test_df["id"].copy()

# 2) Clean data (aturan sama dengan bagian Data Preparation, di-fit pada full train)
X_full = X_full.drop(columns=all_missing_cols, errors="ignore")
X_sub = X_sub.drop(columns=all_missing_cols, errors="ignore")

keep_mask = ~X_full.duplicated(keep="first")
X_full = X_full.loc[keep_mask].reset_index(drop=True)
y_full = y_full.loc[keep_mask].reset_index(drop=True)

full_bounds = {}
for col in X_full.select_dtypes(include=np.number).columns:
    q1, q3 = X_full[col].quantile(0.25), X_full[col].quantile(0.75)
    iqr = q3 - q1
    full_bounds[col] = (q1 - CFG.OUTLIER_IQR_FACTOR * iqr, q3 + CFG.OUTLIER_IQR_FACTOR * iqr)

X_full = clip_outliers(X_full, full_bounds)
X_sub = clip_outliers(X_sub, full_bounds)
X_full = correct_data_errors(X_full)
X_sub = correct_data_errors(X_sub)

# 3) Construct data
X_full = construct_attributes(X_full)
X_sub = construct_attributes(X_sub)
X_sub = X_sub.reindex(columns=X_full.columns)

# 4) Preprocessing pipeline (fit hanya pada full train)
full_num_cols = X_full.select_dtypes(include=np.number).columns.tolist()
full_cat_cols = X_full.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

final_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), full_num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), full_cat_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)
X_full_arr = final_preprocessor.fit_transform(X_full)
X_sub_arr = final_preprocessor.transform(X_sub)

if CFG.APPLY_PCA:
    final_pca = PCA(n_components=CFG.PCA_VARIANCE, random_state=CFG.SEED)
    X_full_arr = final_pca.fit_transform(X_full_arr)
    X_sub_arr = final_pca.transform(X_sub_arr)

X_full_arr = np.asarray(X_full_arr, dtype=float)
X_sub_arr = np.asarray(X_sub_arr, dtype=float)
y_full_arr = y_full.to_numpy()

# 5) Fit model final
if FINAL_MODEL == "knn":
    final_model = KNN(k=KNN_K, distance_metric=KNN_DISTANCE_METRIC, weights=KNN_WEIGHTS)
elif FINAL_MODEL.startswith("svm_"):
    kernel_name = FINAL_MODEL.split("_", 1)[1]
    final_model = SVM(
        kernel=kernel_name, C=1.0, degree=3, gamma="scale", coef0=1.0,
        lr=0.01 if kernel_name == "linear" else 0.005,
        epochs=50, max_kernel_samples=2000, random_state=CFG.SEED,
    )
else:
    raise ValueError(f"FINAL_MODEL tidak dikenal: {FINAL_MODEL}")

final_model.fit(X_full_arr, y_full_arr)
final_model.save_model(str(CFG.OUTPUT_DIR / "models" / f"final_{FINAL_MODEL}_model.pkl"))

# 6) Prediksi dan tulis real_submission.csv (kolom: id, is_canceled)
test_pred = final_model.predict(X_sub_arr)
submission = pd.DataFrame({"id": sub_ids.to_numpy(), CFG.TARGET: test_pred.astype(int)})
submission.to_csv(SUBMISSION_PATH, index=False)

# 7) Sanity check
check = pd.read_csv(SUBMISSION_PATH, dtype={"id": str})
assert list(check.columns) == ["id", CFG.TARGET]
assert len(check) == len(test_df) and check["id"].is_unique
assert check[CFG.TARGET].isin([0, 1]).all() and not check.isna().any().any()

print(f"Submission tersimpan di {SUBMISSION_PATH} ({check.shape[0]} baris)")
display(check.head())
display(check[CFG.TARGET].value_counts(normalize=True).to_frame("proporsi"))


---

# Error Analysis <a name="11"></a>

---

The analysis supports each step taken in this notebook.